In [ ]:
#Pandas para montar tabelas
import pandas as pandas

# Para Salvar e carregar o pipeline de treinamento
import joblib

# Criar Pasta/arquivo
from pathlib import Path

# Ferramentas do Sklearn
from sklearn.preprocessing import MinMaxScaler, OneHotEnconder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import pipeline

In [ ]:
# Classe que vai inserir o lugar onde sera salvo o modelo, a pipeline monta o passo a passo, e 
# feature_names sao os nomes que serao mostrados no final. Fit é onde realmente aprende
class MiniFeatureEngineer:
    def __init__(self, model_dir="models"):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)
        self.path = self.model_dir / "mini_transformer.joblib" # Salvar como mini_transformer.joblib

        # fit() ou load()
        self.pipeline = None
        self.feature_names = None

    # df = Dataframe, num_cols = Colunas, cat_cols = Colunas de categoria
    def fit(self, df, num_cols, cat_cols):
        pre = ColumnTransformer(
            transformer=[
                ("num", MinMaxScaler(), num_cols)
                ("cat", OneHotEnconder(handle_unknown="ignore", sparse_output=False), cat_cols)
            ]
        )

        self.pipeline = Pipeline(steps=["preprocessor", pre])

        # O fit vai aprender o min e max desses dados, e as categorias 
        self.pipeline.fit(df)

        cat = self.pipeline.named_steps["preprocessor"].named_transformers_["cat"]
        self.feature_names = num_cols + cat.get_feature_names_out(cat_cols).tolist()

    # Função para transformar esses dados
    def transform(self, df):
        if self.pipeline in None:
            self.load()
        
        # Transform = aplica as regras aprendiidas no fit, sem reaprender
        data = self.pipeline.transform(df)
        return pd.Dataframe(data, columns=self.feature_names)

    # Salva tudo que for aprendido no fit 
    def save(self):
        joblib.dump({"pipeline": self.pipeline, "feature_names": self.feature_names}, self.path)

    def load(self):
        state = joblib.load(self.path)
        self.pipeline = state["pipeline"]
        self.feature_names = state["feature_names"]


In [ ]:
df_novo = pd.Dataframe([
    {"tempo_entrega":28, "valor_pedido": 27.00, "tipo_entrega": "Bike"}
])

fe_prod = MiniFeatureEngineer()
fe_prod.load()
df_transformado = fe_prod.transform(df_novo)

df_transformado